# LiNGAM + Decision Analysis (Wine Quality) on Localstack

Walkthrough: **Wine Quality (red wine)** dataset, run **LiNGAM**, then run **DA explain** with a non-trivial quality target.

## Prepare data

Dataset: UCI Wine Quality (red wine).

- Features are physicochemical measurements (acidity, sugar, sulphates, alcohol, etc.).
- Target is `quality` (integer score).
- For DA, we set a non-trivial target and choose a row that currently does not satisfy it.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

_tmp = Path(tempfile.mkdtemp(prefix="wine_lingam_"))
WINE_CSV = _tmp / "winequality_red.csv"

wine_df = pd.read_csv(WINE_URL, sep=";")
wine_df.to_csv(WINE_CSV, index=False)

print("shape:", wine_df.shape)
print(wine_df.head().to_string())
print("CSV for upload:", WINE_CSV)

## API URL and API key

Configure local API URL and API key from `examples/.env` or environment variables.

In [ ]:
import sys
from pathlib import Path


def _examples_dir_with_helpers() -> Path:
    cwd = Path.cwd().resolve()
    for anchor in [cwd, *cwd.parents[:12]]:
        for rel in (
            Path("MVP/causal_ai_sdk/examples/helpers.py"),
            Path("causal_ai_sdk/examples/helpers.py"),
        ):
            p = anchor / rel
            if p.is_file():
                return p.parent
    raise ImportError(
        "Cannot find MVP/causal_ai_sdk/examples/helpers.py. Open causal_ai repo as workspace."
    )


sys.path.insert(0, str(_examples_dir_with_helpers().resolve()))

from helpers import get_api_key_from_env, get_base_url_from_env

BASE_URL = get_base_url_from_env()
if not BASE_URL:
    raise ValueError("No API base URL. Set CAUSAL_AI_BASE_URL.")
API_KEY = get_api_key_from_env()
if not API_KEY:
    raise ValueError("No API key. Set CAUSAL_AI_API_KEY.")
masked_key = f"{API_KEY[:6]}...{API_KEY[-4:]}" if len(API_KEY) > 12 else "[provided]"
print("BASE_URL:", BASE_URL)
print("API_KEY:", masked_key)

#### Client imports

In [ ]:
import copy
import json
import math
from typing import Any, Dict

import httpx
from causal_ai_sdk import CausalAIClient

### Step 1 - Create session

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    session = await client.kg.init_session()
    SESSION_UUID = session["uuid"]
print("SESSION_UUID:", SESSION_UUID)

### Step 2 - Upload wine CSV for LiNGAM

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    UPLOAD_INFO = await client.cd.upload_data_for_lingam(SESSION_UUID, WINE_CSV)
print(UPLOAD_INFO.model_dump_json(indent=2))

### Step 3 - Run LiNGAM (submit and wait)

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    cd_task = await client.cd.run_lingam(UPLOAD_INFO, threshold=0.01)
    CD_TASK_ID = cd_task["task_id"]
    print("CD_TASK_ID:", CD_TASK_ID)
    await client.cd.wait_for_task(
        CD_TASK_ID, session_uuid=SESSION_UUID, timeout=600, interval=5
    )
print("LiNGAM completed.")

### Step 4 - Fetch CD result JSON

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    task_result = await client.cd.get_task_result(SESSION_UUID, CD_TASK_ID)
    url = task_result.get("result_url")
    if not url:
        raise RuntimeError("No CD result_url")
    async with httpx.AsyncClient() as http:
        r = await http.get(url)
        r.raise_for_status()
        CD_RESULT = r.json()
if CD_RESULT.get("status") != "succeeded":
    raise RuntimeError(f"CD status: {CD_RESULT.get('status')}")

_cd_preview = copy.deepcopy(CD_RESULT)
for _k in ("uuid", "task_id", "mode", "status", "timestamp"):
    _cd_preview.pop(_k, None)

_cd_preview["adjacency_matrices"] = (_cd_preview.get("adjacency_matrices"))[:2]
_td = _cd_preview["training_data"]
_td["data"] = (_td.get("data"))[:2]

print(json.dumps(_cd_preview, indent=2, default=str))

### Step 5 - EDA for target design

We use a **4 -> medium band** goal:

- pick a row with `quality == 4`
- target `quality in [5, 7]`

This keeps the start point strict (low-quality row) while allowing DA to solve into a realistic medium range.

In [ ]:
q = wine_df["quality"]
print("Rows:", len(wine_df))
print("quality quantiles:")
print(q.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).to_string())

print("\nP(quality == 4):", (q == 4).mean())
print("P(quality in [5,7]):", ((q >= 5) & (q <= 7)).mean())

# 4 -> medium-band objective
QUALITY_TARGET_BAND = [5.0, 7.0]
TARGET_SENSE = "in"

candidate_idx = wine_df.index[wine_df["quality"] == 4].tolist()
if not candidate_idx:
    raise ValueError("No rows found with quality == 4.")
ROW_INDEX = int(candidate_idx[0])

print(
    "\nChosen DA setup: "
    f"quality in {QUALITY_TARGET_BAND}, row_index={ROW_INDEX}, "
    f"current quality={wine_df.loc[ROW_INDEX, 'quality']}"
)

### Step 6 - DA explain (submit and wait)

Target and constraints used in this demo:

- target: `quality in [5, 7]` (move a low row from 4 into medium band)
- immutable: `quality` (do not intervene directly on outcome)
- other features are left unconstrained for demonstration (you can add `increase_only` / `decrease_only` later).

In [ ]:
def _row_to_observation(cd_result: Dict[str, Any], row_index: int = 0) -> Dict[str, float]:
    names = cd_result.get("feature_names") or []
    rows = (cd_result.get("training_data") or {}).get("data") or []
    if not names or not rows:
        raise ValueError("CD result missing feature_names or training_data.data")
    if row_index < 0 or row_index >= len(rows):
        raise IndexError(f"row_index {row_index} out of range (n={len(rows)})")

    row = rows[row_index]
    if len(row) != len(names):
        raise ValueError("Row length does not match feature_names")

    obs: Dict[str, float] = {}
    for name, value in zip(names, row):
        obs[name] = 0.0 if isinstance(value, float) and math.isnan(value) else float(value)
    return obs


current_observation = _row_to_observation(CD_RESULT, row_index=ROW_INDEX)
targets = [{"col": "quality", "sense": TARGET_SENSE, "threshold": QUALITY_TARGET_BAND}]
constraints = {"immutable": ["quality"]}
cd_ref = {"session_uuid": SESSION_UUID, "task_id": CD_TASK_ID}

print("ROW_INDEX:", ROW_INDEX)
print("Current quality:", current_observation.get("quality"))
print("Targets:", targets)
print("Constraints:", constraints)

async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    da_task = await client.da.run_explain(
        session_uuid=SESSION_UUID,
        cd_result_reference=cd_ref,
        current_observation=current_observation,
        targets=targets,
        constraints=constraints,
    )
    DA_TASK_ID = da_task["task_id"]
    print("DA_TASK_ID:", DA_TASK_ID)
    await client.da.wait_for_task(SESSION_UUID, DA_TASK_ID, timeout=300, interval=5)
print("DA completed.")

### Step 7 - Fetch DA result, map action, and show predicted quality uplift

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    da_meta = await client.da.get_task_result(SESSION_UUID, DA_TASK_ID)
    da_url = da_meta.get("result_url")
    if not da_url:
        raise RuntimeError("No DA result_url")
    async with httpx.AsyncClient() as http:
        r = await http.get(da_url)
        r.raise_for_status()
        da_result = r.json()

inner = da_result.get("result") or {}
feature_names = CD_RESULT.get("feature_names") or []
action = inner.get("action") or []

print("is_solved:", inner.get("is_solved"))
print("action:", action)

if len(action) == len(feature_names):
    action_map = {
        name: delta
        for name, delta in zip(feature_names, action)
        if abs(float(delta)) > 1e-12
    }
    print("non-zero action by feature:", action_map)
else:
    print("Cannot map action to feature names (length mismatch).")

# Show predicted improvement level on target (if backend returns it).
current_quality = current_observation.get("quality")
y_cf = inner.get("y_cf")
pred_quality = None

if isinstance(y_cf, (list, tuple)) and y_cf:
    flat = []
    for item in y_cf:
        if isinstance(item, (int, float)):
            flat.append(float(item))
        elif isinstance(item, (list, tuple)) and item:
            v = item[0]
            if isinstance(v, (int, float)):
                flat.append(float(v))
    if flat:
        pred_quality = sum(flat) / len(flat)

if pred_quality is None:
    for key in ("predicted_value", "target_value", "quality", "value"):
        v = inner.get(key)
        if isinstance(v, (int, float)):
            pred_quality = float(v)
            break

print("current quality:", current_quality)
if pred_quality is not None:
    print("predicted quality after action (model estimate):", round(pred_quality, 4))
    print("estimated uplift:", round(pred_quality - float(current_quality), 4))
    in_band = QUALITY_TARGET_BAND[0] <= pred_quality <= QUALITY_TARGET_BAND[1]
    print("predicted in target band [5,7]:", in_band)
else:
    print("predicted quality after action: not provided by backend result schema")

### Step 8 - Cleanup

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    if DA_TASK_ID:
        try:
            await client.da.delete_task(SESSION_UUID, DA_TASK_ID)
            print("DA task deleted")
        except Exception as e:
            print("DA delete:", e)
    if CD_TASK_ID:
        try:
            await client.cd.delete_task(SESSION_UUID, CD_TASK_ID)
            print("CD task deleted")
        except Exception as e:
            print("CD delete:", e)
    try:
        await client.kg.delete_session(SESSION_UUID)
        print("Session deleted")
    except Exception as e:
        print("Session delete:", e)